# Premier entraînement réel — baseline échographie (BUSI)

**Objectif :** valider toute la chaîne (données réelles → entraînement → évaluation → journalisation) avec le modèle le plus simple : ResNet-18 sur BUSI, pli 0.

Ce n'est **pas** encore le criblage d'architectures — c'est le test de plomberie. Si ce notebook tourne de bout en bout, la Phase 2 peut commencer.

## Avant de lancer
1. Le **zip du projet** doit être sur votre Google Drive (la cellule 1 le trouve automatiquement). Le zip est la bonne approche : lire des milliers d'images directement depuis Drive est ~10× plus lent que depuis le disque local de Colab. On dézippe localement à chaque session ; seuls les checkpoints et résultats repartent vers Drive.
2. Ouvrir CE notebook dans Colab : `colab.research.google.com` → Importer le .ipynb (il est aussi dans le zip, `notebooks/`).
3. `Exécution → Modifier le type d'exécution → GPU` (T4 suffit largement ici).
4. Exécuter les cellules dans l'ordre. Durée totale : **~20 min** sur T4 (dont 2-3 min de dézippage).

## Résultat attendu
AUC ≥ 0,90 sur le pli de validation. La littérature sur BUSI dépasse 0,95, mais rappel de la limite documentée : la correspondance image→patiente de BUSI n'est pas publiée, donc une partie des scores très élevés publiés est probablement de la fuite. Notre chiffre, plus prudent, est le bon.

In [ ]:
# ── 1. Montage Drive + décompression du projet vers le disque local ──────
# Le zip reste sur Drive (la source). On décompresse vers /content : le
# disque LOCAL de Colab, bien plus rapide que le montage Drive pour lire
# des milliers d'images pendant l'entraînement. Les checkpoints, eux,
# repartent vers Drive pour survivre aux coupures de session.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, os, zipfile, time

DRIVE = Path('/content/drive/MyDrive')
zips = sorted(DRIVE.glob('projet*.zip')) or sorted(DRIVE.glob('*breast*multimodal*.zip'))
assert zips, f"Aucun projet*.zip trouvé dans {DRIVE} — vérifiez le nom/emplacement"
ZIP = zips[0]
print('Zip trouvé :', ZIP, f'({ZIP.stat().st_size/1e9:.2f} Go)')

REPO = Path('/content/breast-multimodal-fusion')
if not REPO.exists():
    t0 = time.time()
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/_unzip')
    candidates = list(Path('/content/_unzip').rglob('data/manifest.csv'))
    assert candidates, 'manifest.csv introuvable dans le zip — projet complet ?'
    candidates[0].parents[1].rename(REPO)
    print(f'Décompressé en {time.time()-t0:.0f}s')

os.chdir(REPO); sys.path.insert(0, str(REPO))

# Checkpoints et résultats sur DRIVE (survivent aux coupures)
CKPT_DIR = DRIVE / 'breast-multimodal-checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = DRIVE / 'breast-multimodal-results'; RESULTS_DIR.mkdir(exist_ok=True)

print('Dépôt local :', REPO)
print('Manifest    :', (REPO/'data/manifest.csv').exists())
print('Checkpoints :', CKPT_DIR, '(sur Drive)')


In [ ]:
# ── 2. Installation (torch est préinstallé sur Colab) ───────────────────
%pip -q install timm albumentations grad-cam --no-warn-script-location

import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — activer le GPU !')
assert torch.cuda.is_available(), 'Exécution → Modifier le type d\'exécution → GPU'


In [ ]:
# ── 3. Garde-fous AVANT tout entraînement ────────────────────────────────
# Les tests critiques (fuite patiente, fusion) doivent passer ici même.
# Un entraînement lancé sur des splits fuités produit des chiffres faux :
# 2 minutes de tests contre des heures de GPU gâchées.
!python -m pytest tests/test_no_patient_leakage.py tests/test_fusion.py -q --no-header 2>&1 | tail -4


In [ ]:
# ── 4. Données : BUSI, pli 0 ─────────────────────────────────────────────
import pandas as pd, yaml
from src.data.datasets import BreastDataset, build_transforms
from src.data.splits import assert_no_patient_leakage
from torch.utils.data import DataLoader

cfg = yaml.safe_load(open('configs/data.yaml'))
manifest = pd.read_csv('data/manifest.csv')

# Sécurité : si le zip date d'avant la génération des splits, on les recrée
# ici (déterministe, seed 42 — mêmes plis qu'en local).
if 'fold' not in manifest.columns:
    from src.data.splits import make_holdout, make_folds
    dev, test = make_holdout(manifest, 0.15, seed=42)
    dev = make_folds(dev, 5, seed=42)
    test = test.copy(); test['fold'] = -1
    manifest = pd.concat([dev, test], ignore_index=True)
    manifest.to_csv('data/manifest.csv', index=False)
    print('Splits régénérés (le manifeste du zip ne les contenait pas)')

busi = manifest[(manifest.dataset == 'busi') & (manifest.fold >= 0)]  # fold -1 = hold-out, on n'y touche pas
train_df = busi[busi.fold != 0].reset_index(drop=True)
val_df   = busi[busi.fold == 0].reset_index(drop=True)

# Ceinture et bretelles : re-vérifier la fuite sur ce split précis.
assert_no_patient_leakage({'train': train_df, 'val': val_df})

ROOT = REPO / 'data/raw'
train_ds = BreastDataset(train_df, ROOT, build_transforms(cfg, train=True))
val_ds   = BreastDataset(val_df,   ROOT, build_transforms(cfg, train=False))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'train : {len(train_ds)} images ({train_df.label.mean():.1%} malignes)')
print(f'val   : {len(val_ds)} images ({val_df.label.mean():.1%} malignes)')


In [ ]:
# ── 5. Coup d'œil aux données (obligatoire avant d'entraîner) ────────────
# Regarder ce que le modèle va voir attrape les erreurs de prétraitement
# qu'aucun test automatique ne voit (images noires, masques décalés...).
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, i in zip(axes.ravel(), np.random.default_rng(0).choice(len(train_ds), 12, replace=False)):
    s = train_ds[int(i)]
    img = s['image'].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())   # dé-normalise pour l'affichage
    ax.imshow(img); ax.axis('off')
    ax.set_title('malin' if s['label'] > 0.5 else 'bénin', fontsize=9)
plt.suptitle('Échantillon du train APRÈS augmentations — vérifier visuellement')
plt.tight_layout(); plt.show()


In [ ]:
# ── 6. Entraînement : ResNet-18, 25 époques max ──────────────────────────
from src.models.encoders import MonoModalClassifier
from src.train.trainer import Trainer, TrainConfig

model = MonoModalClassifier(encoder_name='resnet18', pretrained=True, image_size=224)
print(f'Paramètres : {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

tcfg = TrainConfig(
    epochs=25, lr=3e-4, batch_size=32, patience=6,
    run_name='busi_resnet18_fold0', out_dir=CKPT_DIR, seed=42,
)
trainer = Trainer(model, train_loader, val_loader, tcfg)
summary = trainer.fit()


In [ ]:
# ── 7. Évaluation complète au point de fonctionnement clinique ───────────
from src.eval.metrics import compute_metrics, bootstrap_ci, specificity_at_sensitivity

trainer.load_best()
scores, targets = trainer.predict(val_loader)

m = compute_metrics(targets, scores)
print('AUC-ROC       :', m['auc_roc'])
print('Sensibilité   :', m['sensitivity'], '| Spécificité :', m['specificity'], f"(seuil Youden {m['threshold']})")
print('F1 / MCC      :', m['f1'], '/', m['mcc'])
print()
for s in (0.90, 0.95, 0.98):
    r = specificity_at_sensitivity(targets, scores, s)
    print(f"À {s:.0%} de sensibilité → spécificité {r['specificity']:.1%}")

# IC bootstrap groupé par patiente (ici 1 image = 1 patiente pour BUSI,
# mais on garde la forme correcte pour les autres datasets)
ci = bootstrap_ci(targets, scores, groups=val_df.patient_id.values, n_boot=1000)
print(f"\nAUC = {ci['value']} , IC95% [{ci['ci_low']} ; {ci['ci_high']}]")


In [ ]:
# ── 8. Courbes : apprentissage + ROC ─────────────────────────────────────
from sklearn.metrics import roc_curve

hist = pd.DataFrame(trainer.history)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(hist.epoch, hist.train_loss, label='train')
a1.plot(hist.epoch, hist.val_loss, label='val')
a1.set_xlabel('époque'); a1.set_ylabel('loss'); a1.legend(); a1.set_title('Apprentissage')

fpr, tpr, _ = roc_curve(targets, scores)
a2.plot(fpr, tpr, label=f"AUC = {m['auc_roc']}")
a2.plot([0, 1], [0, 1], 'k--', alpha=0.3)
a2.set_xlabel('1 - spécificité'); a2.set_ylabel('sensibilité'); a2.legend(); a2.set_title('ROC — BUSI pli 0')
plt.tight_layout(); plt.show()


In [ ]:
# ── 9. Journalisation — chaque entraînement laisse une trace ─────────────
from src.train.trainer import log_run

log_run(
    RESULTS_DIR, 'busi_resnet18_fold0',
    config={'encoder': 'resnet18', 'dataset': 'busi', 'fold': 0,
            'image_size': 224, 'epochs_max': 25, 'seed': 42},
    metrics=m,
    extra={'auc_ci_low': ci['ci_low'], 'auc_ci_high': ci['ci_high'],
           'minutes': summary['minutes']},
)
pd.read_csv(RESULTS_DIR / 'runs.csv').tail(3)


In [ ]:
# ── 10. Bonus : les VRAIS comptes de paramètres (correction du CdC §6.2) ─
from src.models.encoders import print_model_comparison
tbl = print_model_comparison()
print(tbl.to_string(index=False))
tbl.to_csv(RESULTS_DIR / 'model_params.csv', index=False)
print('\n→ À reporter dans le cahier des charges pour corriger le tableau §6.2')


## Lecture des résultats

| Observation | Interprétation |
|---|---|
| AUC ≥ 0,90 | ✅ Chaîne validée — passer au criblage complet (Phase 2, étage 1) |
| AUC 0,80–0,90 | Correct pour un premier tir ; vérifier les courbes de loss (sur-apprentissage ?) avant d'ajuster |
| AUC < 0,80 | Problème de données ou de prétraitement, pas d'architecture — revoir la cellule 5, les images affichées |
| val_loss remonte tôt | Normal sur 519 images ; l'early stopping gère. Si violent : augmenter le dropout |

**Ensuite :** dupliquer ce notebook en changeant `encoder_name` pour le criblage (étage 1 du protocole en entonnoir), puis les 5 plis pour l'étage 2. Chaque exécution s'ajoute à `results/runs.csv`.